# 03 — DataFrame Fundamentals

**Environment:** Ubuntu 22.04 · Python 3.12.3 · OpenJDK 17 · Spark 4.1.x · local PySpark.

Run cells in order. Each notebook can start a Spark session independently.

**Learning goal:** Select, filter, derive, cast safely, handle nulls and remove duplicate transaction IDs.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'transactions.csv').is_file()), None)
assert ROOT is not None, 'Open the workshop folder in Jupyter before running this notebook.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training_lib import create_spark, read_raw, typed_transactions, quality_condition, clean_transactions, completed_enriched, revenue_by_category
from pyspark.sql import functions as F
spark = create_spark('TG117-Workshop')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| master:', spark.sparkContext.master)

In [ ]:
raw, customers, products = read_raw(spark, ROOT)
raw.select('transaction_id', 'customer_id', 'quantity', 'status').show(8, truncate=False)
raw.filter(F.col('status') == 'COMPLETED').show(5, truncate=False)

### Derived columns and safe casts

Spark 4.1 commonly uses ANSI behavior; malformed strings can raise errors with ordinary `cast`. The SQL `try_cast` returns `NULL` instead of failing for malformed input.

In [ ]:
typed = typed_transactions(raw)
typed.select('transaction_id','transaction_date','order_date','quantity','qty','unit_price','price').orderBy('transaction_id').show(12, truncate=False)

### Identify duplicate keys and reject invalid rows

In [ ]:
duplicates = raw.groupBy('transaction_id').count().filter(F.col('count') > 1)
duplicates.show()
invalid = typed.filter(~quality_condition())
invalid.select('transaction_id','transaction_date','customer_id','product_id','quantity','unit_price').orderBy('transaction_id').show(truncate=False)
print('Raw:',raw.count(),'distinct keys:',typed.count(),'invalid after dedup:',invalid.count())
assert (raw.count(),typed.count(),invalid.count()) == (58,57,7)

### Clean dataset and calculate line totals

In [ ]:
clean = clean_transactions(raw)
clean.select('transaction_id','qty','price','line_total').orderBy('transaction_id').show(8, truncate=False)
assert clean.count() == 50

### Participant exercise

1. Show only transactions with quantity greater than 2. 2. Count rows by `status`. 3. Find any rows with null parsed dates. 4. Explain why a transaction with an unknown product ID is **not yet** rejected.

In [ ]:
# Start here:
clean.filter(F.col('qty') > 2).select('transaction_id','qty').show(10)
clean.groupBy('status').count().show()
print('Null parsed dates in cleaned data:', clean.filter(F.col('order_date').isNull()).count())